# Phase 8 — Controlled Model Improvement

This notebook implements the approved improvement path. It investigates the train/test gap, compares a conservative Random Forest with a regularized XGBoost using a validation split from training data only, and evaluates the selected model on the untouched test set once.

No SMOTE, threshold optimization, SHAP, or deployment is performed. The existing models are not overwritten.

## 1. Load the existing artifacts

In [ ]:
import time
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

project = Path.cwd().parent
processed = project / "data" / "processed"
models_dir = project / "models"
results_dir = project / "results"
X_train = sparse.load_npz(processed / "X_train_processed.npz")
X_test = sparse.load_npz(processed / "X_test_processed.npz")
y_train = pd.read_csv(processed / "y_train.csv")["isFraud"].to_numpy()
y_test = pd.read_csv(processed / "y_test.csv")["isFraud"].to_numpy()
old_rf = joblib.load(models_dir / "random_forest.pkl")
old_xgb = joblib.load(models_dir / "xgboost.pkl")
phase6 = pd.read_csv(results_dir / "model_comparison.csv")
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Existing models loaded: Random Forest and XGBoost")
print("Class ratio in train:", (y_train == 0).sum() / (y_train == 1).sum())

## 2. Investigate the gap and create a validation set

In [ ]:
# The final test set remains untouched during model selection.
train_pos, valid_pos = train_test_split(np.arange(len(y_train)), test_size=0.20, random_state=42, stratify=y_train)
X_fit, X_valid = X_train[train_pos], X_train[valid_pos]
y_fit, y_valid = y_train[train_pos], y_train[valid_pos]
print("Fit rows:", len(y_fit), "Validation rows:", len(y_valid))
print("Fit fraud %:", y_fit.mean() * 100, "Validation fraud %:", y_valid.mean() * 100)
print("Test set held out until final evaluation: True")
print("Phase 7 observed RF train-test F1 gap: 0.3323")
print("Phase 7 observed RF train-test PR-AUC gap: 0.2775")

## 3. Time-order drift audit

In [ ]:
merged_path = project / "data" / "processed" / "merged_data.csv.gz"
merged = pd.read_csv(merged_path, compression="gzip", usecols=["TransactionDT", "isFraud"], low_memory=False)
# Recreate the Phase 4 stratified index order to align TransactionDT with X_train rows.
all_idx = np.arange(len(merged))
full_train_idx, full_test_idx = train_test_split(all_idx, test_size=0.20, random_state=42, stratify=merged["isFraud"].to_numpy())
train_dt = merged.iloc[full_train_idx]["TransactionDT"].to_numpy()
train_y_original = merged.iloc[full_train_idx]["isFraud"].to_numpy()
order = np.argsort(train_dt)
ordered_y = train_y_original[order]
chunks = np.array_split(np.arange(len(order)), 5)
time_rows = []
for i, chunk in enumerate(chunks, 1):
    time_rows.append({"Time block": i, "Start TransactionDT": int(train_dt[order[chunk]].min()), "End TransactionDT": int(train_dt[order[chunk]].max()), "Fraud rate %": float(ordered_y[chunk].mean() * 100), "Rows": len(chunk)})
time_table = pd.DataFrame(time_rows)
display(time_table)
print("This is a drift audit only; no time-based model is fit in this cell.")

## 4. Define a common evaluation function

In [ ]:
def evaluate(name, model, X, y):
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return {"Model": name, "Precision": precision_score(y, pred, zero_division=0), "Recall": recall_score(y, pred, zero_division=0), "F1": f1_score(y, pred, zero_division=0), "ROC-AUC": roc_auc_score(y, proba), "PR-AUC": average_precision_score(y, proba), "TN": tn, "FP": fp, "FN": fn, "TP": tp}

validation_results = []
print("Existing Random Forest is kept as a test-set reference only; it is not used for validation selection because it was trained on all Phase 4 training rows.")

## 5. Train a conservative Random Forest

In [ ]:
conservative_rf = RandomForestClassifier(
    n_estimators=20,
    max_depth=8,
    min_samples_leaf=10,
    min_samples_split=20,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
start = time.time()
conservative_rf.fit(X_fit, y_fit)
rf_time = time.time() - start
rf_valid = evaluate("Conservative Random Forest", conservative_rf, X_valid, y_valid)
rf_valid["Training seconds"] = rf_time
validation_results.append(rf_valid)
print(rf_valid)
print("Training seconds:", rf_time)

## 6. Train a regularized XGBoost candidate

In [ ]:
scale_pos_weight = (y_fit == 0).sum() / (y_fit == 1).sum()
regularized_xgb = XGBClassifier(
    n_estimators=30,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=2.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
    tree_method="hist"
)
start = time.time()
regularized_xgb.fit(X_fit, y_fit)
xgb_time = time.time() - start
xgb_valid = evaluate("Regularized XGBoost", regularized_xgb, X_valid, y_valid)
xgb_valid["Training seconds"] = xgb_time
validation_results.append(xgb_valid)
print(xgb_valid)
print("Training seconds:", xgb_time)

## 7. Select a candidate using validation data only

In [ ]:
validation_table = pd.DataFrame(validation_results)
display(validation_table[["Model", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC", "FP", "FN"]].sort_values("PR-AUC", ascending=False))
# PR-AUC is primary for the imbalanced problem; F1 is the secondary tie-breaker.
selected_name = validation_table.sort_values(["PR-AUC", "F1"], ascending=False).iloc[0]["Model"]
selected_validation = validation_table.loc[validation_table["Model"] == selected_name].iloc[0]
print("Selected from validation only:", selected_name)
print("Selection rule: highest validation PR-AUC, then validation F1.")

## 8. Retrain only the selected candidate on all training data and evaluate Test once

In [ ]:
if selected_name == "Conservative Random Forest":
    final_model = RandomForestClassifier(
        n_estimators=20, max_depth=8, min_samples_leaf=10, min_samples_split=20,
        max_features="sqrt", class_weight="balanced", random_state=42, n_jobs=-1
    )
elif selected_name == "Regularized XGBoost":
    final_model = XGBClassifier(
        n_estimators=30, max_depth=3, learning_rate=0.05, min_child_weight=10,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=2.0,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(), random_state=42,
        eval_metric="logloss", n_jobs=-1, tree_method="hist"
    )
else:
    final_model = old_rf
start = time.time()
final_model.fit(X_train, y_train)
final_train_time = time.time() - start
final_test = evaluate(selected_name + " — Final Test", final_model, X_test, y_test)
print(final_test)
print("Final training seconds:", final_train_time)
print("Test was used only after candidate selection: True")

## 9. Compare against the original Phase 6 result

In [ ]:
old_row = phase6.loc[phase6["Model"] == "Random Forest"].iloc[0]
comparison = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"],
    "Phase 6 Random Forest": [old_row["Precision"], old_row["Recall"], old_row["F1"], old_row["ROC-AUC"], old_row["PR-AUC"]],
    "Improved Final Test": [final_test["Precision"], final_test["Recall"], final_test["F1"], final_test["ROC-AUC"], final_test["PR-AUC"]]
})
comparison["Difference"] = comparison["Improved Final Test"] - comparison["Phase 6 Random Forest"]
display(comparison)
print("The original Phase 6 model files were not overwritten.")

## 10. Save only the selected improved model and experiment results

In [ ]:
models_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)
model_path = models_dir / "improved_model.pkl"
results_path = results_dir / "improvement_comparison.csv"
joblib.dump(final_model, model_path)
all_results = pd.concat([validation_table, pd.DataFrame([final_test])], ignore_index=True)
all_results.to_csv(results_path, index=False)
print("Saved model:", model_path)
print("Saved results:", results_path)
print("No threshold optimization, SMOTE, SHAP, or deployment performed.")

## Final Improvement Report

In [ ]:
print("Chosen model:", selected_name)
print("Validation PR-AUC:", selected_validation["PR-AUC"])
print("Final Test Precision:", final_test["Precision"])
print("Final Test Recall:", final_test["Recall"])
print("Final Test F1:", final_test["F1"])
print("Final Test ROC-AUC:", final_test["ROC-AUC"])
print("Final Test PR-AUC:", final_test["PR-AUC"])
print("Final Test TN/FP/FN/TP:", final_test["TN"], final_test["FP"], final_test["FN"], final_test["TP"])
print("Decision: Candidate selected using validation data only; final test evaluated once.")
print("Next step: threshold and calibration analysis only after user review.")

**STOP:** The controlled improvement experiment is complete. The original models remain available, and no threshold optimization or calibration has been performed.